# 04g — Features reutilizadas del master de churn

Recalcula 56 REUSABLE + 10 REFORMULAR sobre el sample de gustos (114,412).
Lógica adaptada de los notebooks 02c/02e/02f/02g/02i/02l/02m del churn:
se retira CUTOFF, se usa REFERENCE_DATE.


In [1]:
import pandas as pd
import numpy as np
import re, time
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_RAW = ROOT / 'data' / 'data_raw'
DATA_QC_GUSTOS = ROOT / 'data' / 'data_qc_gustos'
INFORMES = ROOT / 'informes' / 'fase4_gustos' / '02_features'
DATA_QC_GUSTOS.mkdir(parents=True, exist_ok=True)

REFERENCE_DATE = pd.Timestamp('2026-04-04', tz='UTC')

OID_RE = re.compile(r'[0-9a-f]{24}')
def clean_id(s):
    if pd.isna(s): return None
    m = OID_RE.search(str(s))
    return m.group(0) if m else None

sample = pd.read_parquet(DATA_QC_GUSTOS / 'sample_user_ids_gustos.parquet')
sample_ids = set(sample['user_id'])
N = len(sample)
print(f'Sample N: {N:,}')

t0 = time.time()
features = pd.DataFrame({'user_id': sample['user_id'].values})
print(f'Setup OK')


Sample N: 114,412
Setup OK


In [2]:
# Demografía + estado básico desde users.csv
print('[users] cargando...')
t = time.time()
users = pd.read_csv(DATA_RAW/'users.csv', low_memory=False)
users['user_id'] = users['_id'].apply(clean_id)
users = users[users['user_id'].isin(sample_ids)].set_index('user_id')

features['country'] = features['user_id'].map(users['country'].to_dict())
features['language'] = features['user_id'].map(users['language'].to_dict())
features['user_store_where_published'] = features['user_id'].map(users['store_where_published'].to_dict())
features['user_account_age_days'] = features['user_id'].map(sample.set_index('user_id')['account_age_days'].to_dict())
features['has_user_rated_app'] = features['user_id'].map(users['has_user_rated_app'].astype(str).to_dict())
print(f'[users] OK {time.time()-t:.1f}s')


[users] cargando...


[users] OK 5.1s


In [3]:
# characters: 16 features de char_* (clase, niveles, talent, arena, items, etc.)
print('[characters] cargando...')
t = time.time()
chars = pd.read_csv(DATA_RAW/'characters.csv', low_memory=False)
chars['user_id'] = chars['user_id'].apply(clean_id)
chars = chars[chars['user_id'].isin(sample_ids)].copy()
print(f'  filtrados: {len(chars):,}')

# Map class to int (for char_class_main); ignore NaN classes
class_map = {c: i for i, c in enumerate(chars['class'].dropna().unique())}
chars['class_code'] = chars['class'].map(class_map)

agg = chars.groupby('user_id').agg(
    char_count=('_id','size'),
    char_classes_unique=('class','nunique'),
    char_class_main=('class_code', lambda s: s.mode().iloc[0] if len(s.mode())>0 else np.nan),
    char_level_max=('level','max'),
    char_level_mean=('level','mean'),
    char_experience_max=('experience','max'),
    char_experience_mean=('experience','mean'),
    char_attack_max=('c_total_attack','max'),
    char_critical_chance_max=('c_total_critical_chance','max'),
    char_medals_max=('medals','max'),
    char_arena_category_max=('arena_category','max'),
)
for c in ['char_count','char_classes_unique','char_class_main','char_level_max','char_level_mean',
          'char_experience_max','char_experience_mean','char_attack_max','char_critical_chance_max',
          'char_medals_max','char_arena_category_max']:
    features[c] = features['user_id'].map(agg[c].to_dict())

features['char_has_multiple'] = (features['char_count'] > 1).astype(int)
features['char_has_arena'] = (features['char_arena_category_max'] > 0).astype(int)

# Cosmetic score: cuenta de campos cosméticos no-NaN
cosmetic_cols = ['hairIcon','beardIcon','eyebrowsIcon','faceScarIcon','facePaintIcon','torsoScarIcon','torsoPaintIcon']
chars['cosmetic_score'] = chars[cosmetic_cols].notna().sum(axis=1)
features['char_cosmetic_score'] = features['user_id'].map(chars.groupby('user_id')['cosmetic_score'].max().to_dict())

# Equip slots filled: contar columnas *EqObj no nulas/no vacías por personaje, tomar max
eq_cols = ['rHandEqObj','lHandEqObj','helmEqObj','chestEqObj','handsEqObj','legsEqObj']
chars['eq_filled'] = chars[eq_cols].apply(lambda r: sum(pd.notna(v) and str(v).strip() not in ('','null','None') for v in r), axis=1)
features['char_equip_slots_filled_max'] = features['user_id'].map(chars.groupby('user_id')['eq_filled'].max().to_dict())

# Talent pct spent (per user, max across chars)
chars['talent_pct'] = (chars['spent_talent_points'] / chars['total_talent_points'].replace(0, np.nan)).clip(upper=1.0)
features['char_talent_pct_spent'] = features['user_id'].map(chars.groupby('user_id')['talent_pct'].max().to_dict())

del chars
print(f'[characters] OK {time.time()-t:.1f}s')


[characters] cargando...


  filtrados: 162,895


[characters] OK 21.4s


In [4]:
# items: 9 features (chunked, 17M filas)
print('[items] cargando chunks...')
t = time.time()
agg_dict = {}
for chunk in pd.read_csv(DATA_RAW/'user_items.csv', chunksize=2_000_000, low_memory=False):
    chunk['user_id'] = chunk['user_id'].apply(clean_id)
    chunk = chunk[chunk['user_id'].isin(sample_ids)]
    if len(chunk) == 0: continue
    g = chunk.groupby('user_id').agg(
        n=('_id','size'),
        enhance_sum=('enhance_level','sum'),
        enhance_max=('enhance_level','max'),
        enhance_mean=('enhance_level','mean'),
        attack_mean=('c_base_attack_enhanced','mean'),
        defense_mean=('c_base_defense_enhanced','mean'),
        critical_mean=('c_base_critical_chance','mean'),
        unique_def=('item_definition_excel_id','nunique'),
    )
    for u, row in g.iterrows():
        if u not in agg_dict:
            agg_dict[u] = row.to_dict()
            agg_dict[u]['n_chunks'] = 1
        else:
            agg_dict[u]['n'] += row['n']
            agg_dict[u]['enhance_sum'] += row['enhance_sum']
            agg_dict[u]['enhance_max'] = max(agg_dict[u]['enhance_max'], row['enhance_max'])
            # mean values: weighted average requires sums, use simple max-iter merge
            agg_dict[u]['enhance_mean'] = (agg_dict[u]['enhance_mean']*agg_dict[u]['n_chunks'] + row['enhance_mean']) / (agg_dict[u]['n_chunks']+1)
            agg_dict[u]['attack_mean'] = (agg_dict[u]['attack_mean']*agg_dict[u]['n_chunks'] + row['attack_mean']) / (agg_dict[u]['n_chunks']+1)
            agg_dict[u]['defense_mean'] = (agg_dict[u]['defense_mean']*agg_dict[u]['n_chunks'] + row['defense_mean']) / (agg_dict[u]['n_chunks']+1)
            agg_dict[u]['critical_mean'] = (agg_dict[u]['critical_mean']*agg_dict[u]['n_chunks'] + row['critical_mean']) / (agg_dict[u]['n_chunks']+1)
            agg_dict[u]['unique_def'] = max(agg_dict[u]['unique_def'], row['unique_def'])
            agg_dict[u]['n_chunks'] += 1

items_df = pd.DataFrame.from_dict(agg_dict, orient='index')
items_df.index.name = 'user_id'
features['items_total_instances'] = features['user_id'].map(items_df['n'].to_dict())
features['items_total_enhance_invested'] = features['user_id'].map(items_df['enhance_sum'].to_dict())
features['items_max_enhance_level'] = features['user_id'].map(items_df['enhance_max'].to_dict())
features['items_mean_enhance_level'] = features['user_id'].map(items_df['enhance_mean'].to_dict())
features['items_mean_attack'] = features['user_id'].map(items_df['attack_mean'].to_dict())
features['items_mean_defense'] = features['user_id'].map(items_df['defense_mean'].to_dict())
features['items_mean_critical'] = features['user_id'].map(items_df['critical_mean'].to_dict())
features['items_attack_defense_ratio'] = features['items_mean_attack'] / features['items_mean_defense'].replace(0, np.nan)
features['items_redundancy_ratio'] = features['items_total_instances'] / features['user_id'].map(items_df['unique_def'].to_dict()).replace(0, np.nan)
del agg_dict, items_df
print(f'[items] OK {time.time()-t:.1f}s')


[items] cargando chunks...


[items] OK 37.6s


In [5]:
# collection: 3 features estables (total, unique families, span)
print('[collection] cargando...')
t = time.time()
coll = pd.read_csv(DATA_RAW/'user_items_collection.csv', low_memory=False)
coll['user_id'] = coll['user_id'].apply(clean_id)
coll = coll[coll['user_id'].isin(sample_ids)].copy()
coll['ts'] = pd.to_datetime(coll['created_at'], errors='coerce', utc=True)

agg = coll.groupby('user_id').agg(
    total=('_id','size'),
    unique=('item_definition_excel_id','nunique'),
    first_dt=('ts','min'),
    last_dt=('ts','max'),
)
features['coll_total_items'] = features['user_id'].map(agg['total'].to_dict())
features['coll_unique_families'] = features['user_id'].map(agg['unique'].to_dict())
agg['span_days'] = (agg['last_dt'] - agg['first_dt']).dt.days
features['coll_collection_span_days'] = features['user_id'].map(agg['span_days'].to_dict())

# Reformuladas: coll_age_days (REF - first), pct_days_with_collection_event_60d
agg['age_days'] = (REFERENCE_DATE - agg['first_dt']).dt.days
features['coll_age_days'] = features['user_id'].map(agg['age_days'].to_dict())

# pct_days con evento en últimos 60d
coll['day'] = coll['ts'].dt.date
coll_60d = coll[coll['ts'] >= REFERENCE_DATE - pd.Timedelta(days=60)]
days_active = coll_60d.groupby('user_id')['day'].nunique()
features['pct_days_with_collection_event_60d'] = (features['user_id'].map(days_active.to_dict()) / 60.0).fillna(0).clip(upper=1.0)

# coll_items_recent_30d, coll_items_recent_90d
coll_30 = coll[coll['ts'] >= REFERENCE_DATE - pd.Timedelta(days=30)].groupby('user_id').size()
coll_90 = coll[coll['ts'] >= REFERENCE_DATE - pd.Timedelta(days=90)].groupby('user_id').size()
features['coll_items_recent_30d'] = features['user_id'].map(coll_30.to_dict()).fillna(0)
features['coll_items_recent_90d'] = features['user_id'].map(coll_90.to_dict()).fillna(0)
del coll
print(f'[collection] OK {time.time()-t:.1f}s')


[collection] cargando...


[collection] OK 20.5s


In [6]:
# devices: 5 features (platforms, n models, multi)
print('[devices] cargando...')
t = time.time()
dev = pd.read_csv(DATA_RAW/'devices.csv', low_memory=False)
dev['user_id'] = dev['user_id'].apply(clean_id)
dev = dev[dev['user_id'].isin(sample_ids)].copy()

g = dev.groupby('user_id').agg(
    n_devices=('_id','size'),
    n_models=('device_model','nunique'),
    has_android=('platform', lambda s: int(any(str(x).lower()=='android' for x in s))),
    has_ios=('platform', lambda s: int(any(str(x).lower()=='ios' for x in s))),
)
features['device_records_total'] = features['user_id'].map(g['n_devices'].to_dict())
features['device_unique_models'] = features['user_id'].map(g['n_models'].to_dict())
features['device_has_android'] = features['user_id'].map(g['has_android'].to_dict()).fillna(0).astype(int)
features['device_has_ios'] = features['user_id'].map(g['has_ios'].to_dict()).fillna(0).astype(int)
features['device_is_multi_platform'] = ((features['device_has_android']==1) & (features['device_has_ios']==1)).astype(int)

# Primary platform: el más reciente
dev['updated_dt'] = pd.to_datetime(dev['updated_at'], errors='coerce', utc=True)
latest = dev.sort_values('updated_dt').groupby('user_id').tail(1).set_index('user_id')
features['device_primary_platform'] = features['user_id'].map(latest['platform'].to_dict())
del dev
print(f'[devices] OK {time.time()-t:.1f}s')


[devices] cargando...


[devices] OK 4.9s


In [7]:
# daily_rewards: 8 features
print('[rewards] cargando...')
t = time.time()
rw = pd.read_csv(DATA_RAW/'user_daily_rewards.csv', low_memory=False)
rw['user_id'] = rw['user_id'].apply(clean_id)
rw = rw[rw['user_id'].isin(sample_ids)].copy()

# 1 fila por usuario (snapshot). Si hay duplicados, tomar el último por updated_at.
rw['updated_dt'] = pd.to_datetime(rw['updated_at'], errors='coerce', utc=True)
rw = rw.sort_values('updated_dt').groupby('user_id').tail(1).set_index('user_id')

features['reward_records_total'] = features['user_id'].map((rw.assign(one=1)['one']).to_dict()).fillna(0).astype(int)
features['reward_sets_completed_max'] = features['user_id'].map(rw['sets_completed'].to_dict())
features['reward_current_day_max'] = features['user_id'].map(rw['current_day'].to_dict())
features['reward_premium_days_max'] = features['user_id'].map(rw['last_claimed_premium_reward_day'].to_dict())
features['reward_ad_days_max'] = features['user_id'].map(rw['last_claimed_ad_reward_day'].to_dict())
features['reward_has_premium'] = (features['reward_premium_days_max'] > 0).astype(int)
features['reward_has_ad'] = (features['reward_ad_days_max'] > 0).astype(int)

# reward_claim_rate: proxy = last_claimed_reward_day / current_day (si current_day > 0)
features['reward_claim_rate'] = features['reward_current_day_max'].notna().astype(int) * (
    features['user_id'].map(rw['last_claimed_reward_day'].to_dict()) / features['reward_current_day_max'].replace(0, np.nan)
).clip(upper=1.0)

# reward_register_age_days (REF - created)
rw['created_dt'] = pd.to_datetime(rw['created_at'], errors='coerce', utc=True)
features['reward_register_age_days'] = features['user_id'].map(((REFERENCE_DATE - rw['created_dt']).dt.days).to_dict())
del rw
print(f'[rewards] OK {time.time()-t:.1f}s')


[rewards] cargando...


[rewards] OK 2.2s


In [8]:
# iaps: 13 features (mayoría NaN o 0 por baja cobertura)
print('[iaps] cargando...')
t = time.time()
cons = pd.read_csv(DATA_RAW/'processed_consumables_iaps.csv', low_memory=False)
cons['user_id'] = cons['user_id'].apply(clean_id)
cons = cons[cons['user_id'].isin(sample_ids)].copy()

subs = pd.read_csv(DATA_RAW/'processed_subscriptions_iaps.csv', low_memory=False)
subs['user_id'] = subs['user_id'].apply(clean_id)
subs = subs[subs['user_id'].isin(sample_ids)].copy()
subs['end_dt'] = pd.to_datetime(subs['end_date'], unit='s', utc=True, errors='coerce')
subs['purchase_dt'] = pd.to_datetime(subs['purchase_time'], unit='s', utc=True, errors='coerce')
cons['purchase_dt'] = pd.to_datetime(cons['purchase_time'], unit='s', utc=True, errors='coerce')

cons_users = set(cons['user_id'])
subs_users = set(subs['user_id'])
features['iap_has_consumables'] = features['user_id'].isin(cons_users).astype(int)
features['iap_has_subscription_ever'] = features['user_id'].isin(subs_users).astype(int)
features['iap_is_payer'] = (features['iap_has_consumables'] | features['iap_has_subscription_ever']).astype(int)

features['iap_consumables_count'] = features['user_id'].map(cons.groupby('user_id').size().to_dict()).fillna(0).astype(int)
features['iap_consumables_unique_products'] = features['user_id'].map(cons.groupby('user_id')['product_id'].nunique().to_dict()).fillna(0).astype(int)
features['iap_subscriptions_count'] = features['user_id'].map(subs.groupby('user_id').size().to_dict()).fillna(0).astype(int)
features['iap_has_annual'] = features['user_id'].map(subs[subs['product_id'].astype(str).str.contains('year|annual', case=False, na=False)].groupby('user_id').size().gt(0).astype(int).to_dict()).fillna(0).astype(int)
features['iap_has_quarterly'] = features['user_id'].map(subs[subs['product_id'].astype(str).str.contains('quarter|3month', case=False, na=False)].groupby('user_id').size().gt(0).astype(int).to_dict()).fillna(0).astype(int)
features['iap_trial_only'] = features['user_id'].map(subs.groupby('user_id')['is_trial'].mean().eq(1.0).astype(int).to_dict()).fillna(0).astype(int)

# Reformuladas sobre REFERENCE (no cutoff)
features['iap_subscription_active_now'] = features['user_id'].map((subs.groupby('user_id')['end_dt'].max() >= REFERENCE_DATE).astype(int).to_dict()).fillna(0).astype(int)
features['iap_subscription_active_last_7d_at_ref'] = features['user_id'].map((subs.groupby('user_id')['end_dt'].max() >= (REFERENCE_DATE - pd.Timedelta(days=7))).astype(int).to_dict()).fillna(0).astype(int)
_a30 = features['user_id'].map(cons[cons['purchase_dt'] >= REFERENCE_DATE - pd.Timedelta(days=30)].groupby('user_id').size().gt(0).to_dict()).fillna(False).astype(bool)
_b30 = features['user_id'].map(subs[subs['purchase_dt'] >= REFERENCE_DATE - pd.Timedelta(days=30)].groupby('user_id').size().gt(0).to_dict()).fillna(False).astype(bool)
features['iap_paid_last_30d_at_ref'] = (_a30 | _b30).astype(int)
_a90 = features['user_id'].map(cons[cons['purchase_dt'] >= REFERENCE_DATE - pd.Timedelta(days=90)].groupby('user_id').size().gt(0).to_dict()).fillna(False).astype(bool)
_b90 = features['user_id'].map(subs[subs['purchase_dt'] >= REFERENCE_DATE - pd.Timedelta(days=90)].groupby('user_id').size().gt(0).to_dict()).fillna(False).astype(bool)
features['iap_paid_last_90d_at_ref'] = (_a90 | _b90).astype(int)
features['iap_is_recent_payer'] = (features['iap_paid_last_90d_at_ref'] == 1).astype(int)
del cons, subs
print(f'[iaps] OK {time.time()-t:.1f}s')


[iaps] cargando...


[iaps] OK 1.3s


In [9]:
# feedback: 4 features (sin feedback_has_any por baja varianza)
print('[feedback] cargando...')
t = time.time()
fb = pd.read_csv(DATA_RAW/'support_user_feedback_by_type.csv', low_memory=False)
fb['user_id'] = fb['user_id'].apply(clean_id)
fb = fb[fb['user_id'].isin(sample_ids)].copy()

features['feedback_total'] = features['user_id'].map(fb.groupby('user_id').size().to_dict()).fillna(0).astype(int)

# Clasificar por feedback_type (heurística por keyword)
fb['type_lower'] = fb['feedback_type'].astype(str).str.lower()
fb['is_positive'] = fb['type_lower'].str.contains('positive|like|love|great|happy', na=False).astype(int)
fb['is_negative'] = fb['type_lower'].str.contains('negative|hate|bad|crash|bug|frustr', na=False).astype(int)
fb['is_monetization'] = fb['type_lower'].str.contains('pay|buy|purchase|monet|price|refund', na=False).astype(int)

features['feedback_n_positive'] = features['user_id'].map(fb.groupby('user_id')['is_positive'].sum().to_dict()).fillna(0).astype(int)
features['feedback_n_negative'] = features['user_id'].map(fb.groupby('user_id')['is_negative'].sum().to_dict()).fillna(0).astype(int)
features['feedback_n_monetization'] = features['user_id'].map(fb.groupby('user_id')['is_monetization'].sum().to_dict()).fillna(0).astype(int)
del fb
print(f'[feedback] OK {time.time()-t:.1f}s')


[feedback] cargando...


[feedback] OK 0.3s


In [10]:
# Save + sanity checks
assert len(features) == N, f'N mismatch: {len(features)} vs {N}'
assert features['user_id'].is_unique
features.to_parquet(DATA_QC_GUSTOS / 'features_reutilizadas_churn.parquet', index=False)
print(f'\nGuardado: {len(features):,} filas × {features.shape[1]} cols')
print(f'Tiempo total: {time.time()-t0:.1f}s')
print('\nResumen NaN (top 15):')
print((features.isna().mean()*100).sort_values(ascending=False).head(15).round(2))
print('\nShape:', features.shape)

# Reporte
rep_dir = INFORMES / '04g_reutilizadas'
rep_dir.mkdir(parents=True, exist_ok=True)
nan_rates = (features.isna().mean()*100).round(2)
report = [f'# 04g - Features reutilizadas del churn\n', f'**Fecha**: {time.strftime("%Y-%m-%d %H:%M")}\n',
          f'**Shape**: {features.shape}\n', f'**Tiempo**: {time.time()-t0:.1f}s\n', '\n## NaN por feature\n']
report += ['| Feature | %NaN |', '|---|---:|']
for f, v in nan_rates.sort_values(ascending=False).items():
    report.append(f'| {f} | {v} |')
(rep_dir / 'execution_report.md').write_text('\n'.join(report))
print(f'\nReporte guardado')



Guardado: 114,412 filas × 71 cols
Tiempo total: 93.5s

Resumen NaN (top 15):
device_unique_models          2.80
device_primary_platform       2.80
device_records_total          2.80
char_talent_pct_spent         2.75
char_arena_category_max       1.82
reward_current_day_max        0.49
reward_register_age_days      0.49
reward_claim_rate             0.49
reward_ad_days_max            0.49
reward_premium_days_max       0.49
reward_sets_completed_max     0.49
user_store_where_published    0.01
has_user_rated_app            0.00
coll_total_items              0.00
coll_unique_families          0.00
dtype: float64

Shape: (114412, 71)

Reporte guardado
